# Feature Analysis for Explainable Credit-Risk Modelling

This notebook audits the available Home Credit predictors and develops research-oriented candidate feature sets. The categories and statuses are analytical aids, not legally authoritative classifications or final feature-selection decisions. No modelling, preprocessing, encoding, or modification of the raw data is performed.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 1. Load Data

In [2]:
data_path = "../data/raw/application_train.csv"
df = pd.read_csv(data_path)
predictor_columns = df.columns.drop("TARGET")
column_dtypes = df.dtypes
numerical_features = df.select_dtypes(include=np.number).columns.drop("TARGET", errors="ignore").tolist()
categorical_features = predictor_columns.difference(numerical_features, sort=False).tolist()

print(f"Dataset shape: {df.shape}")
print(f"Total predictors (excluding TARGET): {len(predictor_columns)}")
print(f"Numerical features: {len(numerical_features)}")
print(f"Categorical features: {len(categorical_features)}")

Dataset shape: (307511, 122)
Total predictors (excluding TARGET): 121
Numerical features: 105
Categorical features: 16


## 2. Build the Feature Audit Table

Feature categories below are heuristic research groupings derived from names and should be manually reviewed.

In [3]:
def categorise_feature(feature):
    if feature == "SK_ID_CURR":
        return "Identifier"
    if feature.startswith("EXT_SOURCE_"):
        return "External Credit Score"
    if feature.startswith("AMT_REQ_CREDIT_BUREAU_"):
        return "Credit Bureau Request"
    if feature.startswith("FLAG_DOCUMENT_") or feature.startswith("FLAG_"):
        return "Document/Flag"
    if feature.startswith(("REGION_", "REG_", "LIVE_REGION_", "LIVE_CITY_")):
        return "Geographic/Region"
    if feature.endswith(("_AVG", "_MODE", "_MEDI")) or feature in {"TOTALAREA_MODE", "FONDKAPREMONT_MODE", "HOUSETYPE_MODE", "WALLSMATERIAL_MODE", "EMERGENCYSTATE_MODE"}:
        return "Property/Building"
    if feature in {"CODE_GENDER", "DAYS_BIRTH", "NAME_FAMILY_STATUS", "CNT_CHILDREN"}:
        return "Demographic"
    if feature in {"DAYS_EMPLOYED", "OCCUPATION_TYPE", "ORGANIZATION_TYPE", "NAME_INCOME_TYPE", "FLAG_EMP_PHONE", "DAYS_REGISTRATION"}:
        return "Employment"
    if feature in {"AMT_INCOME_TOTAL"}:
        return "Income"
    if feature.startswith(("AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE", "NAME_CONTRACT_", "WEEKDAY_APPR_", "HOUR_APPR_")):
        return "Loan/Application"
    if feature in {"NAME_HOUSING_TYPE", "FLAG_OWN_REALTY", "FLAG_OWN_CAR", "OWN_CAR_AGE"}:
        return "Housing"
    if feature in {"CNT_FAM_MEMBERS", "NAME_TYPE_SUITE"}:
        return "Family"
    if feature in {"FLAG_MOBIL", "FLAG_CONT_MOBILE", "FLAG_PHONE", "FLAG_EMAIL", "FLAG_WORK_PHONE", "DAYS_LAST_PHONE_CHANGE"}:
        return "Contact Information"
    if feature in {"NAME_EDUCATION_TYPE"}:
        return "Demographic"
    return "Other"

# Compute each full-dataset statistic once, then reuse the cached Series.
missing_counts = df.isna().sum()
missing_percentages = missing_counts.div(len(df)).mul(100)
unique_value_counts = df.nunique(dropna=True)

audit = pd.DataFrame({
    "feature": predictor_columns,
    "dtype": column_dtypes.loc[predictor_columns].astype(str).to_numpy(),
    "missing_count": missing_counts.loc[predictor_columns].to_numpy(),
    "missing_percentage": missing_percentages.loc[predictor_columns].to_numpy(),
    "unique_values": unique_value_counts.loc[predictor_columns].to_numpy(),
})
audit["feature_category"] = audit["feature"].map(categorise_feature)
audit["candidate_status"] = "Candidate"
audit["actionability"] = "Review"
audit["notes"] = "Requires manual research review."
audit.head()

,feature,dtype,missing_count,missing_percentage,unique_values,feature_category,candidate_status,actionability,notes
0,SK_ID_CURR,int64,0,0.0,307511,Identifier,Candidate,Review,Requires manual research review.
1,NAME_CONTRACT_TYPE,object,0,0.0,2,Loan/Application,Candidate,Review,Requires manual research review.
2,CODE_GENDER,object,0,0.0,3,Demographic,Candidate,Review,Requires manual research review.
3,FLAG_OWN_CAR,object,0,0.0,2,Document/Flag,Candidate,Review,Requires manual research review.
4,FLAG_OWN_REALTY,object,0,0.0,2,Document/Flag,Candidate,Review,Requires manual research review.


## 3–7. Apply Structural, Sensitivity, Actionability, and Interpretability Flags

In [4]:
def update_features(features, status=None, actionability=None, note=None, category=None):
    mask = audit["feature"].isin(features)
    if status is not None:
        audit.loc[mask, "candidate_status"] = status
    if actionability is not None:
        audit.loc[mask, "actionability"] = actionability
    if note is not None:
        audit.loc[mask, "notes"] = note
    if category is not None:
        audit.loc[mask, "feature_category"] = category

update_features(
    ["SK_ID_CURR"], status="Exclude", actionability="Not applicable",
    category="Identifier",
    note="Unique applicant identifier; should not be used as an explanatory credit-risk factor.",
)

high_missing_mask = audit["missing_percentage"] > 60
audit.loc[high_missing_mask, "candidate_status"] = "Review/Exclude"
audit.loc[high_missing_mask, "notes"] = "More than 60% missing; retain for manual review rather than automatic removal."

sensitive_or_immutable = [
    "CODE_GENDER", "DAYS_BIRTH", "NAME_FAMILY_STATUS", "CNT_CHILDREN",
    "NAME_EDUCATION_TYPE", "CNT_FAM_MEMBERS",
]
update_features(
    sensitive_or_immutable, status="Review", actionability="Non-actionable",
    note="Potentially sensitive or immutable personal characteristic; flag for fairness and actionability review without drawing a legal conclusion.",
)

strong_candidates = [
    "AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE",
    "DAYS_EMPLOYED", "NAME_INCOME_TYPE", "NAME_HOUSING_TYPE",
    "REGION_RATING_CLIENT", "REGION_RATING_CLIENT_W_CITY",
]
update_features(
    strong_candidates, status="Strong candidate", actionability="Partially actionable",
    note="Human-readable financial or application characteristic proposed for explainability-focused modelling review.",
)

bureau_request_features = predictor_columns[predictor_columns.str.startswith("AMT_REQ_CREDIT_BUREAU_")].tolist()
update_features(
    bureau_request_features, status="Candidate", actionability="Partially actionable",
    note="Recent credit-enquiry activity may be understandable with appropriate time-window context.",
)

external_sources = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]
update_features(
    external_sources, status="Review", actionability="Non-actionable",
    category="External Credit Score",
    note="May provide predictive value but is difficult to explain directly to ordinary users without additional context.",
)

actionable_features = ["AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE"]
update_features(actionable_features, actionability="Actionable")
audit

,feature,dtype,missing_count,missing_percentage,unique_values,feature_category,candidate_status,actionability,notes
0,SK_ID_CURR,int64,0,0.000000,307511,Identifier,Exclude,Not applicable,Unique applicant identifier; should not be use...
1,NAME_CONTRACT_TYPE,object,0,0.000000,2,Loan/Application,Candidate,Review,Requires manual research review.
2,CODE_GENDER,object,0,0.000000,3,Demographic,Review,Non-actionable,Potentially sensitive or immutable personal ch...
3,FLAG_OWN_CAR,object,0,0.000000,2,Document/Flag,Candidate,Review,Requires manual research review.
4,FLAG_OWN_REALTY,object,0,0.000000,2,Document/Flag,Candidate,Review,Requires manual research review.
...,...,...,...,...,...,...,...,...,...
116,AMT_REQ_CREDIT_BUREAU_DAY,float64,41519,13.501631,9,Credit Bureau Request,Candidate,Partially actionable,Recent credit-enquiry activity may be understa...
117,AMT_REQ_CREDIT_BUREAU_WEEK,float64,41519,13.501631,9,Credit Bureau Request,Candidate,Partially actionable,Recent credit-enquiry activity may be understa...
118,AMT_REQ_CREDIT_BUREAU_MON,float64,41519,13.501631,24,Credit Bureau Request,Candidate,Partially actionable,Recent credit-enquiry activity may be understa...
119,AMT_REQ_CREDIT_BUREAU_QRT,float64,41519,13.501631,11,Credit Bureau Request,Candidate,Partially actionable,Recent credit-enquiry activity may be understa...


## 8. Repeated Property and Building Feature Families

In [5]:
repeated_mask = audit["feature"].str.endswith(("_AVG", "_MODE", "_MEDI"))
repeated_features = audit.loc[repeated_mask, "feature"]
audit.loc[repeated_mask & ~audit["candidate_status"].eq("Review/Exclude"), "candidate_status"] = "Review"
audit.loc[repeated_mask & ~audit["candidate_status"].eq("Review/Exclude"), "notes"] = (
    "Repeated AVG/MODE/MEDI property measure; review for redundancy before selection."
)

redundancy_map = pd.DataFrame({
    "group/prefix": repeated_features.str.replace(r"_(AVG|MODE|MEDI)$", "", regex=True),
    "feature": repeated_features,
}).merge(
    audit[["feature", "missing_percentage"]], on="feature", how="left"
)
redundancy_summary = (
    redundancy_map.groupby("group/prefix", as_index=False)
    .agg(
        number_of_related_features=("feature", "count"),
        average_missing_percentage=("missing_percentage", "mean"),
    )
    .sort_values(["number_of_related_features", "average_missing_percentage"], ascending=False)
)
redundancy_summary

,group/prefix,number_of_related_features,average_missing_percentage
2,COMMONAREA,3,69.872297
13,NONLIVINGAPARTMENTS,3,69.432963
11,LIVINGAPARTMENTS,3,68.354953
7,FLOORSMIN,3,67.848630
18,YEARS_BUILD,3,66.497784
10,LANDAREA,3,59.376738
1,BASEMENTAREA,3,58.515956
14,NONLIVINGAREA,3,55.179164
3,ELEVATORS,3,53.295980
0,APARTMENTS,3,50.749729


## 9. Proposed Candidate Subsets

These lists are proposals only. Their presence does not imply final inclusion, suitability, or legal acceptability.

In [6]:
interpretable_core_proposal = [
    "NAME_CONTRACT_TYPE", "AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY",
    "AMT_GOODS_PRICE", "DAYS_EMPLOYED", "NAME_INCOME_TYPE",
    "NAME_EDUCATION_TYPE", "NAME_HOUSING_TYPE", "CNT_FAM_MEMBERS",
    "REGION_RATING_CLIENT", "REGION_RATING_CLIENT_W_CITY",
    "AMT_REQ_CREDIT_BUREAU_MON", "AMT_REQ_CREDIT_BUREAU_QRT",
    "AMT_REQ_CREDIT_BUREAU_YEAR",
]
interpretable_core = [f for f in interpretable_core_proposal if f in predictor_columns]

extended_additions = [
    "FLAG_OWN_CAR", "FLAG_OWN_REALTY", "OWN_CAR_AGE", "OCCUPATION_TYPE",
    "ORGANIZATION_TYPE", "NAME_TYPE_SUITE", "WEEKDAY_APPR_PROCESS_START",
    "HOUR_APPR_PROCESS_START", "REGION_POPULATION_RELATIVE",
    "DAYS_REGISTRATION", "DAYS_ID_PUBLISH", "DAYS_LAST_PHONE_CHANGE",
    "FLAG_PHONE", "FLAG_EMAIL", "AMT_REQ_CREDIT_BUREAU_HOUR",
    "AMT_REQ_CREDIT_BUREAU_DAY", "AMT_REQ_CREDIT_BUREAU_WEEK",
    "EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3",
]
extended_explainable = list(dict.fromkeys(
    interpretable_core + [f for f in extended_additions if f in predictor_columns]
))

review_features = audit.loc[
    audit["candidate_status"].isin(["Review", "Review/Exclude"]), "feature"
].tolist()

print(f"Interpretable Core ({len(interpretable_core)} features):")
print(interpretable_core)
print(f"\nExtended Explainable ({len(extended_explainable)} features):")
print(extended_explainable)
print(f"\nReview Features ({len(review_features)} features):")
print(review_features)

Interpretable Core (15 features):
['NAME_CONTRACT_TYPE', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE', 'DAYS_EMPLOYED', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_HOUSING_TYPE', 'CNT_FAM_MEMBERS', 'REGION_RATING_CLIENT', 'REGION_RATING_CLIENT_W_CITY', 'AMT_REQ_CREDIT_BUREAU_MON', 'AMT_REQ_CREDIT_BUREAU_QRT', 'AMT_REQ_CREDIT_BUREAU_YEAR']

Extended Explainable (35 features):
['NAME_CONTRACT_TYPE', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE', 'DAYS_EMPLOYED', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_HOUSING_TYPE', 'CNT_FAM_MEMBERS', 'REGION_RATING_CLIENT', 'REGION_RATING_CLIENT_W_CITY', 'AMT_REQ_CREDIT_BUREAU_MON', 'AMT_REQ_CREDIT_BUREAU_QRT', 'AMT_REQ_CREDIT_BUREAU_YEAR', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'OWN_CAR_AGE', 'OCCUPATION_TYPE', 'ORGANIZATION_TYPE', 'NAME_TYPE_SUITE', 'WEEKDAY_APPR_PROCESS_START', 'HOUR_APPR_PROCESS_START', 'REGION_POPULATION_RELATIVE', 'DAYS_REGISTRATION', 'DAYS_ID_PUBLISH', 'DAYS_LAST_PHONE_CHANGE',

## 10. Summary Tables

In [7]:
display(audit["feature_category"].value_counts().rename_axis("feature_category").to_frame("feature_count"))
display(audit["candidate_status"].value_counts().rename_axis("candidate_status").to_frame("feature_count"))
display(audit["actionability"].value_counts().rename_axis("actionability").to_frame("feature_count"))

,feature_count
feature_category,
Property/Building,47
Document/Flag,28
Geographic/Region,9
Credit Bureau Request,6
Loan/Application,6
Demographic,5
Other,5
Employment,5
External Credit Score,3


,feature_count
candidate_status,
Candidate,54
Review,40
Review/Exclude,17
Strong candidate,9
Exclude,1


,feature_count
actionability,
Review,96
Partially actionable,12
Non-actionable,9
Actionable,3
Not applicable,1


In [8]:
print("Top 20 highest-missing features")
display(audit.nlargest(20, "missing_percentage"))

Top 20 highest-missing features


,feature,dtype,missing_count,missing_percentage,unique_values,feature_category,candidate_status,actionability,notes
47,COMMONAREA_AVG,float64,214865,69.872297,3181,Property/Building,Review/Exclude,Review,More than 60% missing; retain for manual revie...
61,COMMONAREA_MODE,float64,214865,69.872297,3128,Property/Building,Review/Exclude,Review,More than 60% missing; retain for manual revie...
75,COMMONAREA_MEDI,float64,214865,69.872297,3202,Property/Building,Review/Exclude,Review,More than 60% missing; retain for manual revie...
55,NONLIVINGAPARTMENTS_AVG,float64,213514,69.432963,386,Property/Building,Review/Exclude,Review,More than 60% missing; retain for manual revie...
69,NONLIVINGAPARTMENTS_MODE,float64,213514,69.432963,167,Property/Building,Review/Exclude,Review,More than 60% missing; retain for manual revie...
83,NONLIVINGAPARTMENTS_MEDI,float64,213514,69.432963,214,Property/Building,Review/Exclude,Review,More than 60% missing; retain for manual revie...
85,FONDKAPREMONT_MODE,object,210295,68.386172,4,Property/Building,Review/Exclude,Review,More than 60% missing; retain for manual revie...
53,LIVINGAPARTMENTS_AVG,float64,210199,68.354953,1868,Property/Building,Review/Exclude,Review,More than 60% missing; retain for manual revie...
67,LIVINGAPARTMENTS_MODE,float64,210199,68.354953,736,Property/Building,Review/Exclude,Review,More than 60% missing; retain for manual revie...
81,LIVINGAPARTMENTS_MEDI,float64,210199,68.354953,1097,Property/Building,Review/Exclude,Review,More than 60% missing; retain for manual revie...


In [9]:
for status in ["Strong candidate", "Review", "Exclude"]:
    print(f"{status} features")
    display(audit.loc[audit["candidate_status"].eq(status)].reset_index(drop=True))

Strong candidate features


,feature,dtype,missing_count,missing_percentage,unique_values,feature_category,candidate_status,actionability,notes
0,AMT_INCOME_TOTAL,float64,0,0.000000,2548,Income,Strong candidate,Partially actionable,Human-readable financial or application charac...
1,AMT_CREDIT,float64,0,0.000000,5603,Loan/Application,Strong candidate,Actionable,Human-readable financial or application charac...
2,AMT_ANNUITY,float64,12,0.003902,13672,Loan/Application,Strong candidate,Actionable,Human-readable financial or application charac...
3,AMT_GOODS_PRICE,float64,278,0.090403,1002,Loan/Application,Strong candidate,Actionable,Human-readable financial or application charac...
4,NAME_INCOME_TYPE,object,0,0.000000,8,Employment,Strong candidate,Partially actionable,Human-readable financial or application charac...
5,NAME_HOUSING_TYPE,object,0,0.000000,6,Housing,Strong candidate,Partially actionable,Human-readable financial or application charac...
6,DAYS_EMPLOYED,int64,0,0.000000,12574,Employment,Strong candidate,Partially actionable,Human-readable financial or application charac...
7,REGION_RATING_CLIENT,int64,0,0.000000,3,Geographic/Region,Strong candidate,Partially actionable,Human-readable financial or application charac...
8,REGION_RATING_CLIENT_W_CITY,int64,0,0.000000,3,Geographic/Region,Strong candidate,Partially actionable,Human-readable financial or application charac...


Review features


,feature,dtype,missing_count,missing_percentage,unique_values,feature_category,candidate_status,actionability,notes
0,CODE_GENDER,object,0,0.000000,3,Demographic,Review,Non-actionable,Potentially sensitive or immutable personal ch...
1,CNT_CHILDREN,int64,0,0.000000,15,Demographic,Review,Non-actionable,Potentially sensitive or immutable personal ch...
2,NAME_EDUCATION_TYPE,object,0,0.000000,5,Demographic,Review,Non-actionable,Potentially sensitive or immutable personal ch...
3,NAME_FAMILY_STATUS,object,0,0.000000,6,Demographic,Review,Non-actionable,Potentially sensitive or immutable personal ch...
4,DAYS_BIRTH,int64,0,0.000000,17460,Demographic,Review,Non-actionable,Potentially sensitive or immutable personal ch...
5,CNT_FAM_MEMBERS,float64,2,0.000650,17,Family,Review,Non-actionable,Potentially sensitive or immutable personal ch...
6,EXT_SOURCE_1,float64,173378,56.381073,114584,External Credit Score,Review,Non-actionable,May provide predictive value but is difficult ...
7,EXT_SOURCE_2,float64,660,0.214626,119831,External Credit Score,Review,Non-actionable,May provide predictive value but is difficult ...
8,EXT_SOURCE_3,float64,60965,19.825307,814,External Credit Score,Review,Non-actionable,May provide predictive value but is difficult ...
9,APARTMENTS_AVG,float64,156061,50.749729,2339,Property/Building,Review,Review,Repeated AVG/MODE/MEDI property measure; revie...


Exclude features


,feature,dtype,missing_count,missing_percentage,unique_values,feature_category,candidate_status,actionability,notes
0,SK_ID_CURR,int64,0,0.0,307511,Identifier,Exclude,Not applicable,Unique applicant identifier; should not be use...


In [10]:
print("Interpretable Core candidate list")
display(pd.DataFrame({"feature": interpretable_core}))

print("Extended Explainable candidate list")
display(pd.DataFrame({"feature": extended_explainable}))

Interpretable Core candidate list


,feature
0,NAME_CONTRACT_TYPE
1,AMT_INCOME_TOTAL
2,AMT_CREDIT
3,AMT_ANNUITY
4,AMT_GOODS_PRICE
5,DAYS_EMPLOYED
6,NAME_INCOME_TYPE
7,NAME_EDUCATION_TYPE
8,NAME_HOUSING_TYPE
9,CNT_FAM_MEMBERS


Extended Explainable candidate list


,feature
0,NAME_CONTRACT_TYPE
1,AMT_INCOME_TOTAL
2,AMT_CREDIT
3,AMT_ANNUITY
4,AMT_GOODS_PRICE
5,DAYS_EMPLOYED
6,NAME_INCOME_TYPE
7,NAME_EDUCATION_TYPE
8,NAME_HOUSING_TYPE
9,CNT_FAM_MEMBERS


## 11. Save Analysis Output

Executing the following cell saves only the feature audit—not a modified dataset—to `backend/artifacts/feature_audit.csv`.

In [11]:
audit_output_path = "../artifacts/feature_audit.csv"
audit.to_csv(audit_output_path, index=False)
print(f"Feature audit saved to: {audit_output_path}")

Feature audit saved to: ../artifacts/feature_audit.csv


## Research Interpretation

- **Predictive usefulness vs explainability:** _Discuss the trade-off after empirical modelling and explanation evaluation._
- **Missing-data burden:** _Discuss whether heavily missing predictors can be justified and handled transparently._
- **Sensitive/immutable attributes:** _Discuss fairness and research implications without making legal conclusions._
- **Actionability for DiCE:** _Discuss which features could support realistic counterfactual changes and which should remain fixed._
- **Identifier exclusion:** _Explain that identifiers do not represent meaningful explanatory credit-risk factors._
- **Technical or redundant features:** _Discuss why opaque scores and repeated property measures may weaken customer-facing explanations._
- **Performance and regulatory clarity:** _Discuss how predictive performance should be balanced with understandable, transparent communication._

These prompts are for later interpretation and are not final dissertation or legal claims.

## Decision Required Before Modelling

The final feature set must be manually reviewed before any preprocessing or XGBoost training. No model is trained in this notebook.